# 21. 변이 0개 환자의 THYM 쏠림 확인 (OOF)

## 배경
- 05: LR이 변이 0개 환자(모든 유전자가 WT) 94명을 전부 THYM으로 예측 → THYM 과예측 원인으로 지목
- 11/10 회의 할 일: 이 쏠림이 XGBoost와 섞기 모델에서도 나타나는지 OOF로 확인
- 변이 0개 환자는 피처가 **완전히 같은 행** → 어떤 모델이든 모두 같은 클래스로 예측할 수밖에 없음
  - 따라서 질문은 "쏠림이 있는가"가 아니라 **"THYM이 이 행들에 줄 수 있는 최선의 클래스인가"**

## 질문
- Q1: 변이 0개 환자는 train·test에 몇 명이고, train의 실제 라벨은 무엇인가?
- Q2: XGB, 섞기 모델(14, 16)도 이들을 전부 THYM으로 예측하는가?
- Q3: 다른 클래스로 바꾸면 Macro F1이 오르는가? (fold 학습 부분으로 고르고 검증 fold에서 채점)
- Q4: THYM 확률을 낮추는 보정(THYM 과예측 완화)이 도움이 되는가? (같은 방식)

## 검증 방법
- 데이터·모델 설정은 14·16과 동일 (`load_data()` 정규화, 팀 공통 fold)
- 비교 모델: LR(05), XGB(10), 14 섞기(XGB 0.7 + LR 0.3, LB 0.3247), 16 섞기(핫스팟 제외, LB에 낸 select6 비율 RF 5 / LR_trunc 4 / XGB 3 / ET 2 ÷14, LB 0.3183 기각)
- Q3·Q4는 **교차 적합**: fold k의 선택은 나머지 fold의 OOF로만 하고 fold k에서 채점 → 낙관적 점수 방지
- test는 예측 분포 관찰만 (선택 기준에 쓰지 않음)
- 실행 환경: macOS M1 (arm64). XGB가 들어간 점수는 설아님 데스크탑(Windows)과 ±0.004 다를 수 있음 (17 리뷰에서 확인), LR은 동일

#### 준비: 데이터, 모델 4종의 OOF·test 확률 (약 5분)

In [1]:
import sys; sys.path.append('..')
import numpy as np, pandas as pd, xgboost as xgb
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_sample_weight
from src.config import SEED
from src.data import load_data, get_gene_cols
from src.features import mutation_features
from src.validation import seed_everything, encode_target, get_folds, run_cv, macro_f1

seed_everything()
train, test, submission = load_data()
genes = get_gene_cols(train)
y, le = encode_target(train)
CLS = list(le.classes_)
THYM = CLS.index('THYM')
folds = get_folds(y)
F_tr, F_te = mutation_features(train, genes), mutation_features(test, genes)
cat = lambda parts: (pd.concat([F_tr[p] for p in parts], axis=1), pd.concat([F_te[p] for p in parts], axis=1))

class BalancedXGB(xgb.XGBClassifier):
    def fit(self, X, y, **kw):
        return super().fit(X, y, sample_weight=compute_sample_weight('balanced', y), **kw)

lr = lambda: LogisticRegression(C=0.1, max_iter=3000, class_weight='balanced', random_state=SEED)
tree_kw = dict(n_estimators=500, max_features='sqrt', min_samples_leaf=2, class_weight='balanced_subsample',
               random_state=SEED, n_jobs=-1)
MODELS = {   # 14·16과 같은 설정
    'LR':       (cat(['any']), lr),
    'LR_trunc': (cat(['any', 'trunc']), lr),
    'XGB':      (cat(['any', 'stats']), lambda: BalancedXGB(n_estimators=200, learning_rate=0.1, max_depth=3,
                                                             colsample_bytree=0.3, tree_method='hist',
                                                             eval_metric='mlogloss', random_state=SEED, n_jobs=-1)),
    'RF':       (cat(['any', 'stats']), lambda: RandomForestClassifier(**tree_kw)),
    'ET':       (cat(['any', 'stats']), lambda: ExtraTreesClassifier(**tree_kw)),
}
P_oof, P_test = {}, {}
for name, ((X, X_te), mk) in MODELS.items():
    r = run_cv(mk, X, y, X_te, folds=folds, verbose=False)
    P_oof[name], P_test[name] = r['oof_proba'], r['test_proba']
    print(f"{name:9s} CV {r['cv_mean']:.4f} ± {r['cv_std']:.4f}")

BLENDS = {   # 이름: {모델: 비율}
    'LR (05)':        {'LR': 1},
    'XGB (10)':       {'XGB': 1},
    '14 섞기':        {'XGB': 0.7, 'LR': 0.3},
    '16 섞기 (핫스팟 제외)': {'RF': 5 / 14, 'LR_trunc': 4 / 14, 'XGB': 3 / 14, 'ET': 2 / 14},   # LB에 낸 select6 비율 (설아님 데스크탑)
}
mix = lambda w, P: sum(v * P[k] for k, v in w.items())

LR        CV 0.3818 ± 0.0064
LR_trunc  CV 0.3920 ± 0.0075
XGB       CV 0.4125 ± 0.0109
RF        CV 0.3905 ± 0.0100
ET        CV 0.3803 ± 0.0148


#### Q1. 변이 0개 환자 수와 train 실제 라벨

In [2]:
zero_tr = (train[genes] == 'WT').all(axis=1).to_numpy()
zero_te = (test[genes] == 'WT').all(axis=1).to_numpy()
print(f'변이 0개: train {zero_tr.sum()}명 ({zero_tr.mean():.1%}), test {zero_te.sum()}명 ({zero_te.mean():.1%})')

lab = pd.Series(np.array(CLS)[y[zero_tr]]).value_counts()
total = pd.Series(np.array(CLS)[y]).value_counts()
pd.DataFrame({'변이 0개 환자 수': lab, '그 암종 전체': total[lab.index],
              '암종 중 변이 0개 비율': (lab / total[lab.index]).round(3)})

변이 0개: train 94명 (1.5%), test 31명 (1.2%)


,변이 0개 환자 수,그 암종 전체,암종 중 변이 0개 비율
THYM,30,98,0.306
THCA,18,324,0.056
LAML,12,158,0.076
BRCA,5,786,0.006
KIPAN,4,515,0.008
PRAD,4,266,0.015
SARC,3,198,0.015
LGG,3,229,0.013
CESC,3,155,0.019
PCPG,3,147,0.020


#### Q2. 모델별 예측: 변이 0개 환자를 무엇으로 예측하나 + THYM 예측 비율

In [3]:
rows = []
for name, w in BLENDS.items():
    po, pt = mix(w, P_oof).argmax(1), mix(w, P_test).argmax(1)
    rows.append({'모델': name,
                 'OOF: 변이 0개 예측': pd.Series(np.array(CLS)[po[zero_tr]]).value_counts().to_dict(),
                 'test: 변이 0개 예측': pd.Series(np.array(CLS)[pt[zero_te]]).value_counts().to_dict(),
                 'THYM 실제 비율': f'{np.mean(y == THYM):.1%}',
                 'THYM 예측 (OOF)': f'{np.mean(po == THYM):.1%}',
                 'THYM 예측 (test)': f'{np.mean(pt == THYM):.1%}',
                 'THYM F1 (OOF)': round(f1_score(y == THYM, po == THYM), 3)})
pd.DataFrame(rows)

,모델,OOF: 변이 0개 예측,test: 변이 0개 예측,THYM 실제 비율,THYM 예측 (OOF),THYM 예측 (test),THYM F1 (OOF)
0,LR (05),{'THYM': 94},{'THYM': 31},1.6%,10.1%,6.6%,0.172
1,XGB (10),{'THYM': 94},{'THYM': 31},1.6%,4.3%,3.1%,0.278
2,14 섞기,{'THYM': 94},{'THYM': 31},1.6%,4.4%,3.0%,0.268
3,16 섞기 (핫스팟 제외),{'THYM': 94},{'THYM': 31},1.6%,5.2%,3.5%,0.258


#### Q3. 변이 0개 환자를 다른 클래스로 바꾸면? (교차 적합)
- 변이 0개 행은 모두 같은 클래스를 받으므로, 26개 클래스 중 하나를 골라 통째로 바꿔 봄
- fold k: 나머지 fold의 OOF로 Macro F1이 가장 높은 클래스를 고르고 → fold k에서 채점

In [4]:
rows = []
for name in ['14 섞기', '16 섞기 (핫스팟 제외)']:
    pred = mix(BLENDS[name], P_oof).argmax(1)
    full = {CLS[c]: macro_f1(y, np.where(zero_tr, c, pred)) for c in range(len(CLS))}
    top = pd.Series(full).sort_values(ascending=False)
    picks = []
    for tr_idx, _ in folds:
        sc = {c: f1_score(y[tr_idx], np.where(zero_tr[tr_idx], c, pred[tr_idx]), average='macro') for c in range(len(CLS))}
        picks.append(CLS[max(sc, key=sc.get)])
    rows.append({'모델': name, '현재 (THYM)': round(full['THYM'], 4),
                 '2위 클래스': f'{top.index[1]} {top.iloc[1]:.4f}', 'fold별 선택': picks})
pd.DataFrame(rows)

,모델,현재 (THYM),2위 클래스,fold별 선택
0,14 섞기,0.4190,KIPAN 0.4136,"[THYM, THYM, THYM, THYM, THYM]"
1,16 섞기 (핫스팟 제외),0.4274,KIPAN 0.4228,"[THYM, THYM, THYM, THYM, THYM]"


#### Q4. THYM 확률을 낮추면? (교차 적합)
- THYM 확률에 배율(0.3 ~ 1.2)을 곱해 다시 argmax → THYM 과예측을 줄이는 보정
- 배율은 fold 학습 부분으로 고르고 검증 fold에서 채점

In [5]:
SCALES = [0.3, 0.5, 0.7, 0.85, 1.0, 1.2]
rows = []
for name in ['14 섞기', '16 섞기 (핫스팟 제외)']:
    P = mix(BLENDS[name], P_oof)
    def pred(s, idx):
        Q = P[idx].copy(); Q[:, THYM] *= s
        return Q.argmax(1)
    base, adj, picks = [], [], []
    for tr_idx, va in folds:
        s = max(SCALES, key=lambda s: f1_score(y[tr_idx], pred(s, tr_idx), average='macro'))
        picks.append(s)
        base.append(f1_score(y[va], pred(1.0, va), average='macro'))
        adj.append(f1_score(y[va], pred(s, va), average='macro'))
    d = np.array(adj) - np.array(base)
    rows.append({'모델': name, 'fold별 배율': picks, 'CV 보정 전': round(np.mean(base), 4),
                 'CV 보정 후': round(np.mean(adj), 4), '차이': round(d.mean(), 4), '오른 fold': f'{int((d > 0).sum())}/5'})
pd.DataFrame(rows)

,모델,fold별 배율,CV 보정 전,CV 보정 후,차이,오른 fold
0,14 섞기,"[0.5, 0.5, 1.0, 0.7, 1.0]",0.4187,0.4169,-0.0017,0/5
1,16 섞기 (핫스팟 제외),"[1.0, 1.0, 1.0, 0.7, 1.0]",0.4277,0.4262,-0.0016,0/5


## 결과

**Q1. 변이 0개 환자**: train 94명(1.5%), test 31명(1.2%)
- train 실제 라벨: THYM 30, THCA 18, LAML 12, BRCA 5, KIPAN 4, PRAD 4 … (15개 암종)
- THYM은 전체 98명 중 **31%(30명)가 변이 0개** → 변이 0개 비율이 가장 높은 암종 (LAML 7.6%, THCA 5.6%)

**Q2. 모든 모델이 변이 0개 환자를 전부 THYM으로 예측** (LR, XGB, 14 섞기, 16 섞기 모두 OOF 94/94, test 31/31)

| 모델 | THYM 예측 (OOF) | THYM 예측 (test) | THYM F1 (OOF) |
| :--- | :-: | :-: | :-: |
| 14 섞기 | 4.4% | 3.0% | 0.268 |
| 16 섞기 (핫스팟 제외, LB 0.3183 기각) | 5.2% | 3.5% | 0.258 |

- THYM 실제 비율은 1.6% → OOF에서 약 3배 과예측. 변이 0개(94명) 외에 변이가 1~2개뿐인 환자도 THYM으로 많이 감
- test는 3.0~3.5%로 OOF보다 덜함 (test는 환자당 변이가 더 많아 변이 1~2개 환자 비율이 낮음: train 6.0%, test 3.7%)

**Q3. 다른 클래스로 바꾸기**: THYM이 최선
- 14 섞기: THYM 0.4190 > 2위 KIPAN 0.4136 / 16 섞기: THYM 0.4274 > 2위 KIPAN 0.4228
- 교차 적합에서도 5개 fold 모두 THYM을 선택

**Q4. THYM 확률 낮추기**: 오히려 하락
- 14 섞기 −0.0017, 16 섞기 −0.0016, 오른 fold 0/5

## 결론
- **변이 0개 환자의 THYM 예측은 "쏠림 버그"가 아니라 Macro F1 기준 최선의 선택**
  - 피처가 완전히 같은 94명은 한 클래스로만 예측 가능 → 그중 THYM이 30명(32%)으로 가장 많고, THYM 재현율에도 크게 기여
  - 다른 클래스로 바꾸거나 THYM 확률을 낮추면 교차 적합 CV가 모두 내려감
- 모델(LR → XGB → 섞기)을 바꿔도 같은 결과 → 데이터 구조의 한계 (변이 정보가 없으면 구분할 근거가 없음)
- **조치 없음**: 후처리하지 않고 현재 예측 유지
- 최종 후보(14 섞기, LB 0.3247)에서도 같은 결론 → 16은 LB 기각(0.3183)됐지만 비교용으로 남겨 둠
- 실행 환경 참고: 이 노트북은 macOS M1에서 실행. 데스크탑(Windows)에서는 XGB가 들어간 점수가 ±0.004 다를 수 있지만, THYM이 최선이라는 결론은 차이(2위와 0.005 이상)보다 커서 바뀌지 않음
- 발표용 해석: "THYM 과예측은 변이가 없는 환자를 THYM으로 모은 결과이며, 변이 0개 환자 중 THYM이 가장 많아 F1 기준 최선. 변이 데이터만으로는 이 환자들을 구분할 수 없음"